# {{ notebook_title }}

Raw teacher activations before perturbation for ID and OOD datasets.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

sns.set_theme(style="darkgrid", context="notebook")

ROOT = Path.cwd().parent.parent
if not (ROOT / "runs").exists():
    ROOT = ROOT.parent

EXPERIMENT_NAME = "{{ experiment_name }}"
ACTIVATION_DIR = ROOT / "runs" / EXPERIMENT_NAME / "teacher_activations"
MANIFEST_PATH = ACTIVATION_DIR / "manifest.json"
ID_DATASET = "{{ id_dataset }}"

DATASET_LABELS = {{ dataset_labels }}
DATASET_ORDER = {{ dataset_order }}

MANIFEST_PATH


In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Run: sbatch slurm_scripts/export_teacher_activations.sbatch"
    )

with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

artifact_index = {
    (entry["dataset"], entry["layer"]): ROOT / entry["path"]
    for entry in manifest["artifacts"]
}
layers = manifest["layers"]
datasets = [dataset for dataset in DATASET_ORDER if any((dataset, layer) in artifact_index for layer in layers)]

layers, datasets


In [ ]:
def load_activation_artifact(dataset: str, layer: str) -> dict[str, object]:
    """Load one teacher activation artifact."""

    path = artifact_index[(dataset, layer)]
    return torch.load(path, map_location="cpu", weights_only=False)


def mean_activation_frame(layer: str, dataset_keys: list[str]) -> pd.DataFrame:
    """Build a DataFrame of sample-averaged flattened activations."""

    frames = []
    for dataset in dataset_keys:
        artifact = load_activation_artifact(dataset, layer)
        activations = artifact["activations"].flatten(start_dim=1).float()
        means = activations.mean(dim=0)
        frames.append(
            pd.DataFrame(
                {
                    "activation_index": range(means.numel()),
                    "mean_activation": means.numpy(),
                    "dataset": DATASET_LABELS.get(dataset, dataset),
                    "layer": layer,
                }
            )
        )
    return pd.concat(frames, ignore_index=True)


def plot_activation_bars(layer: str, dataset_keys: list[str], max_positions: int | None = 512) -> None:
    """Plot sample-averaged flattened activations before perturbation."""

    data = mean_activation_frame(layer, dataset_keys)
    if max_positions is not None:
        data = data[data["activation_index"] < max_positions]
    height = max(3.0, 1.1 * len(dataset_keys))
    fig, axes = plt.subplots(len(dataset_keys), 1, figsize=(12, height), sharex=True, sharey=True)
    if len(dataset_keys) == 1:
        axes = [axes]
    for axis, dataset in zip(axes, dataset_keys, strict=True):
        label = DATASET_LABELS.get(dataset, dataset)
        subset = data[data["dataset"] == label]
        sns.barplot(
            data=subset,
            x="activation_index",
            y="mean_activation",
            ax=axis,
            color="#4C78A8",
            edgecolor="#4C78A8",
            errorbar=None,
            width=1.0,
        )
        axis.set_title(label)
        axis.set_ylabel("Mean activation")
        axis.tick_params(axis="x", labelbottom=False)
    axes[-1].tick_params(axis="x", labelbottom=True)
    axes[-1].set_xlabel("Flattened activation position")
    step = max(1, data["activation_index"].max() // 16)
    axes[-1].set_xticks(range(0, data["activation_index"].max() + 1, step))
    fig.suptitle(f"{layer} teacher activations before perturbation", y=1.02)
    fig.tight_layout()


def plot_id_ood_overlay(layer: str, ood_dataset: str, max_positions: int | None = 512) -> None:
    """Overlay ID and one OOD dataset after averaging over samples."""

    data = mean_activation_frame(layer, [ID_DATASET, ood_dataset])
    if max_positions is not None:
        data = data[data["activation_index"] < max_positions]
    fig, axis = plt.subplots(figsize=(12, 3.8))
    id_label = DATASET_LABELS[ID_DATASET]
    ood_label = DATASET_LABELS.get(ood_dataset, ood_dataset)
    id_subset = data[data["dataset"] == id_label]
    ood_subset = data[data["dataset"] == ood_label]
    positions = id_subset["activation_index"].to_numpy()
    axis.bar(
        positions - 0.2,
        id_subset["mean_activation"].to_numpy(),
        width=0.4,
        color="#4C78A8",
        edgecolor="#4C78A8",
        linewidth=0,
        label=id_label,
    )
    axis.bar(
        positions + 0.2,
        ood_subset["mean_activation"].to_numpy(),
        width=0.4,
        color="#F58518",
        edgecolor="#F58518",
        linewidth=0,
        label=ood_label,
    )
    step = max(1, data["activation_index"].max() // 16)
    axis.set_xticks(range(0, data["activation_index"].max() + 1, step))
    axis.set_xlabel("Flattened activation position")
    axis.set_ylabel("Mean activation")
    axis.set_title(f"{layer}: ID vs {ood_label} before perturbation")
    axis.legend(title=None)
    fig.tight_layout()



## Layer3

In [ ]:
plot_activation_bars("layer3", datasets, max_positions=None)


In [ ]:
for ood_dataset in [dataset for dataset in datasets if dataset != ID_DATASET]:
    plot_id_ood_overlay("layer3", ood_dataset, max_positions=None)


## Layer4

In [ ]:
if "layer4" in layers:
    plot_activation_bars("layer4", datasets, max_positions=None)


In [ ]:
if "layer4" in layers:
    for ood_dataset in [dataset for dataset in datasets if dataset != ID_DATASET]:
        plot_id_ood_overlay("layer4", ood_dataset, max_positions=None)
